In [1]:
# Cell 1 - Setup
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
# Cell 2 - Verify imports
import statsmodels.api as sm
print('statsmodels:', sm.__version__)

statsmodels: 0.14.6


In [3]:
# Cell 3 - Load processed data
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


In [4]:
# Cell 4 - WMAE metric
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [5]:
# Cell 5 - Build ALL series (full dataset, not a sample)
# No auto_arima search here - per assignment instructions, ARIMA gets light
# training treatment. A fixed (p,d,q) applied across all ~3,331 series keeps
# full-dataset coverage feasible without an hours-long per-series search.
VAL_LEN = 39
MIN_LEN = 20 + VAL_LEN  # need at least some pre-validation history to fit at all

all_series = train.groupby(['Store','Dept']).size().index.tolist()
print('Total Store-Dept combinations:', len(all_series))

series_data = {}
skipped = 0
for idx, (store, dept) in enumerate(all_series):
    sub = train[(train['Store']==store) & (train['Dept']==dept)].sort_values('Date')
    sub = sub.set_index('Date').asfreq('W-FRI')
    sub['Weekly_Sales'] = sub['Weekly_Sales'].interpolate().bfill().ffill()
    sub['IsHoliday'] = sub['IsHoliday'].astype(float).fillna(0.0).astype(bool)

    if len(sub) < MIN_LEN:
        skipped += 1
        continue

    series_data[(store, dept)] = {
        'values': sub['Weekly_Sales'].values,
        'is_holiday': sub['IsHoliday'].values
    }

    if (idx+1) % 500 == 0:
        print(f'Processed {idx+1}/{len(all_series)} series...')

print(f'\nSeries kept: {len(series_data)}  |  Skipped (too short): {skipped}')

Total Store-Dept combinations: 3331
Processed 500/3331 series...
Processed 1000/3331 series...
Processed 1500/3331 series...
Processed 2000/3331 series...
Processed 2500/3331 series...
Processed 3000/3331 series...

Series kept: 3179  |  Skipped (too short): 152


In [6]:
# Cell 6 - 39-week holdout split (same convention as every other notebook)
fit_raw, val_raw, val_holiday = {}, {}, {}
for key, d in series_data.items():
    fit_raw[key] = d['values'][:-VAL_LEN]
    val_raw[key] = d['values'][-VAL_LEN:]
    val_holiday[key] = d['is_holiday'][-VAL_LEN:]

In [7]:
# Cell 7 - Fit ARIMA with a FIXED order across all series (no per-series search)
# order=(1,1,1) is a simple, standard, defensible starting point: 1 AR term,
# 1 differencing to remove trend, 1 MA term. No seasonal component - that
# limitation IS the point of contrast with SARIMA, and is exactly what the
# assignment wants explained theoretically rather than tuned away.
from statsmodels.tsa.arima.model import ARIMA

FIXED_ORDER = (1, 1, 1)

arima_models = {}
all_true, all_pred, all_holiday = [], [], []
failed_keys = []

for i, (key, fseries) in enumerate(fit_raw.items()):
    try:
        model = ARIMA(fseries, order=FIXED_ORDER)
        fitted = model.fit()
        arima_models[key] = fitted
        pred = fitted.forecast(steps=VAL_LEN)
        all_true.extend(val_raw[key])
        all_pred.extend(pred)
        all_holiday.extend(val_holiday[key])
    except Exception:
        failed_keys.append(key)
        continue

    if (i+1) % 500 == 0:
        print(f'Fitted {i+1}/{len(fit_raw)} series...')

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_holiday = np.array(all_holiday)
print(f'\nDone. Fitted: {len(arima_models)}  |  Failed to converge: {len(failed_keys)}')
print('Total validation points:', len(all_true))

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Fitted 500/3179 series...


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Fitted 1000/3179 series...


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


...

Fitted 2500/3179 series...


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Fitted 3000/3179 series...


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Done. Fitted: 3179  |  Failed to converge: 0
Total validation points: 123981


In [8]:
# Cell 8 - Evaluate
arima_wmae = wmae(all_true, all_pred, all_holiday)
print('ARIMA WMAE (full dataset):', arima_wmae)
print('Holiday WMAE:    ', wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday]))
print('Non-holiday WMAE:', wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday]))

ARIMA WMAE (full dataset): 2566.449540654437
Holiday WMAE:     2742.8629307002734
Non-holiday WMAE: 2521.56002855313


In [9]:
# Cell 9 - Baseline for fair comparison (52-week seasonal naive, same full set)
baseline_pred_list = []
for key in arima_models.keys():
    vals = series_data[key]['values']
    fit_vals = vals[:-VAL_LEN]
    if len(fit_vals) >= VAL_LEN+52:
        lookback = fit_vals[-VAL_LEN-52:-52]
    else:
        lookback = np.full(VAL_LEN, fit_vals.mean())
    baseline_pred_list.extend(lookback[:VAL_LEN])

baseline_pred_arr = np.array(baseline_pred_list)
baseline_score = wmae(all_true, baseline_pred_arr, all_holiday)
print('52-week seasonal baseline WMAE:', baseline_score)
print('ARIMA WMAE:                    ', arima_wmae)

52-week seasonal baseline WMAE: 4030.742560174325
ARIMA WMAE:                     2566.449540654437


In [10]:
# Cell 10 - Save models locally
import joblib
os.makedirs('models', exist_ok=True)
joblib.dump(arima_models, 'models/arima_best.pkl')
print('Saved.', len(arima_models), 'fitted series models.')

Saved. 3179 fitted series models.


In [11]:
# Cell 11 - Raw-input pipeline wrapper
from sklearn.base import BaseEstimator, RegressorMixin

class ARIMAPanelForecaster(BaseEstimator, RegressorMixin):
    def __init__(self, models, fallback_median):
        self.models = models
        self.fallback_median = fallback_median

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        df = X[['Store','Dept','Date']].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['_order'] = np.arange(len(df))
        preds = np.empty(len(df), dtype=float)

        for (store, dept), group in df.groupby(['Store','Dept']):
            key = (store, dept)
            group = group.sort_values('Date')
            if key not in self.models:
                preds[group['_order'].to_numpy()] = self.fallback_median
                continue
            n_needed = len(group)
            forecast = self.models[key].forecast(steps=n_needed)
            preds[group['_order'].to_numpy()] = np.asarray(forecast)[:n_needed]

        return preds

In [12]:
# Cell 12 - Wrap fitted models as pipeline
global_median_fallback = float(train['Weekly_Sales'].median())
final_pipeline = ARIMAPanelForecaster(models=arima_models, fallback_median=global_median_fallback)
print('ARIMA pipeline ready |', len(arima_models), 'series covered')

ARIMA pipeline ready | 3179 series covered


In [15]:
# Cell 13 - Sanity check
test_rows = pd.DataFrame({
    'Store': [k[0] for k in list(arima_models.keys())[:20]],
    'Dept':  [k[1] for k in list(arima_models.keys())[:20]],
    'Date':  [pd.Timestamp('2012-11-02')] * 20
})

# FIX: Changed .iloc[0] to [0] because .forecast() returns a numpy array
_direct_pred = np.array([arima_models[k].forecast(steps=1)[0] for k in list(arima_models.keys())[:20]])

_pipe_pred = final_pipeline.predict(test_rows)
pipeline_max_abs_diff = float(np.max(np.abs(_direct_pred - _pipe_pred)))

print('max |pipeline - direct predict| =', pipeline_max_abs_diff)

max |pipeline - direct predict| = 0.0


In [16]:
# Cell 14 - MLflow / DagsHub setup
!pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

import mlflow
import mlflow.sklearn

EXPERIMENT_NAME = 'ARIMA_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)
if mlflow.active_run() is not None:
    mlflow.end_run()

print('Experiment:', EXPERIMENT_NAME)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=11ef6a42-9044-4d0d-b190-80539df7be43&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=eba133de8aa31622a3042e82eaf6a5088d79e24f01fc81cc76eb892b987f752c




Accessing as ggzob23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

2026/07/11 21:17:00 INFO mlflow.tracking.fluent: Experiment with name 'ARIMA_Training' does not exist. Creating a new experiment.


Experiment: ARIMA_Training


In [17]:
# Cell 15 - Run 1: Cleaning
with mlflow.start_run(run_name='ARIMA_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'freq': 'W-FRI',
        'interpolation': 'linear + bfill/ffill',
        'dataset_scope': 'full (all Store-Dept series)',
        'min_series_length': MIN_LEN,
    })
    mlflow.log_metrics({
        'total_store_dept_combinations': len(all_series),
        'series_kept': len(series_data),
        'series_skipped_too_short': skipped,
        'val_weeks': VAL_LEN,
    })

🏃 View run ARIMA_Cleaning at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10/runs/6e84f90f5ac0466289024db64e7ec4a5
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10


In [18]:
# Cell 16 - Run 2: Feature selection reasoning
with mlflow.start_run(run_name='ARIMA_Feature_Selection'):
    mlflow.set_tag('stage', 'feature_selection')
    mlflow.log_params({
        'selection_strategy': 'univariate only - no exogenous regressors',
        'model_input': 'Weekly_Sales history only',
        'reason': 'Plain ARIMA has no built-in mechanism for external regressors '
                   'or seasonality. Per assignment instructions, ARIMA/SARIMA get '
                   'light theoretical treatment rather than heavy engineering.',
    })

🏃 View run ARIMA_Feature_Selection at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10/runs/13d70fac4e4d415188d48837963ee307
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10


In [19]:
# Cell 17 - Run 3: Baseline
with mlflow.start_run(run_name='ARIMA_Baseline'):
    mlflow.set_tag('stage', 'baseline')
    mlflow.log_param('strategy', '52-week seasonal naive, same full dataset as ARIMA')
    mlflow.log_metric('val_wmae', baseline_score)

🏃 View run ARIMA_Baseline at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10/runs/aba4ae596b48450f95c582f81eb7cc28
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10


In [20]:
# Cell 18 - Run 4: Validation
with mlflow.start_run(run_name='ARIMA_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.log_params({
        'scheme': 'chronological holdout, 39-week convention, full dataset',
        'val_weeks': VAL_LEN,
        'order': str(FIXED_ORDER),
        'search_performed': False,
        'reason_no_search': 'Assignment instructs minimal training time for ARIMA; '
                             'a fixed order across all ~3,331 series keeps full-dataset '
                             'coverage feasible without an hours-long per-series search.',
        'n_series_fitted': len(arima_models),
        'n_series_failed': len(failed_keys),
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_score,
        'arima_wmae': arima_wmae,
        'improvement_over_baseline': baseline_score - arima_wmae,
    })

🏃 View run ARIMA_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10/runs/9e11dfa92dcf4fe5a624b50e28b69c07
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10


In [21]:
# Cell 19 - Run 5: Final model + raw-input pipeline (registration DEFERRED)
from mlflow.models import infer_signature

input_example = pd.DataFrame({
    'Store': [k[0] for k in list(arima_models.keys())[:5]],
    'Dept':  [k[1] for k in list(arima_models.keys())[:5]],
    'Date':  [pd.Timestamp('2012-11-02')] * 5,
})
output_example = final_pipeline.predict(input_example)
signature = infer_signature(input_example, output_example)

with mlflow.start_run(run_name='ARIMA_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'arima', 'dataset_scope': 'full'})
    mlflow.log_metrics({
        'final_wmae': arima_wmae,
        'baseline_wmae': baseline_score,
        'pipeline_sanity_max_abs_diff': pipeline_max_abs_diff,
    })

    mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path='pipeline',
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        input_example=input_example,
        signature=signature,
    )
    mlflow.log_artifact('models/arima_best.pkl', artifact_path='estimator')

    FINAL_RUN_ID = final_run.info.run_id
    FINAL_MODEL_URI = f'runs:/{FINAL_RUN_ID}/pipeline'

print('ARIMA final run:', FINAL_RUN_ID)

2026/07/11 21:17:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 21:17:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run ARIMA_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10/runs/48765005258c4bd6903b821082bd2f98
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/10
ARIMA final run: 48765005258c4bd6903b821082bd2f98


In [22]:
# Cell 20 - Register ONLY if ARIMA is the best architecture overall
REGISTER_AS_GLOBAL_BEST = False

if REGISTER_AS_GLOBAL_BEST:
    registered_version = mlflow.register_model(model_uri=FINAL_MODEL_URI, name=REGISTERED_MODEL_NAME)
    print('Registered model:', REGISTERED_MODEL_NAME, '| version:', registered_version.version)
else:
    print('Pipeline logged and saved, not registered. Register only if ARIMA wins overall.')

Pipeline logged and saved, not registered. Register only if ARIMA wins overall.
